In [10]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import matplotlib.dates as mdates
from sklearn import datasets, linear_model
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D
import statsmodels.api as sm
from scipy.stats import t
from scipy.optimize import minimize
import os
import cvxpy as cp
from tqdm import tqdm
import seaborn as sns
import torch

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import openpyxl

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler

# Data Preparation

Do following for different risk aversions beta:

for date in test_dates:

1. prepare train and test data (date)

2. prepare scenario matrix out of train data

3. specify constants for cvxpy (dimensions of scenario matrix, etc.)

4. precompute oracle solution w*(c)

5. Train VAR on train with custom loss function (combinations of MSE and DFL loss)

    Do so with every combination
    Save relevant metrics
    Estimate returns c_hat and w*(c_hat)

-> Agreggate metrics ofer the whole test period (backtesting period)

In [3]:
data_path = "../data/processed/"

return_data = pd.read_csv(data_path + "stationary_return_data_subset.csv")

In [4]:
# Pivot return data
return_matrix = return_data.pivot(index='date', columns='RIC', values='return')
display(return_matrix)

RIC,BRO.N,CTRA.N,CVS.N,FFIV.OQ,HSY.N,LLY.N,NEM.N,PCG.N,REGN.OQ,TYL.N
date,,,,,,,,,,
2000-01-31,-0.104405,-0.081712,-1.236885e-01,-0.175439,-0.105263,0.005639,-0.168367,7.012195e-02,-0.034314,-0.204545
2000-02-29,-0.035086,0.074857,1.788909e-03,-0.042553,0.039651,-0.107465,0.085890,-5.982906e-02,3.588832,0.228571
2000-03-31,0.172348,0.142292,7.321429e-02,-0.247222,0.109531,0.059937,0.015535,3.281437e-02,-0.476770,0.104651
2000-04-28,0.037157,0.027682,1.595897e-01,-0.310886,-0.069231,0.227183,0.044568,2.351190e-01,-0.033827,-0.094737
2000-05-31,0.165747,0.345877,1.398903e-11,-0.309237,0.148974,-0.011941,-0.016000,1.444134e-11,-0.286652,-0.255814
...,...,...,...,...,...,...,...,...,...,...
2023-09-29,-0.057490,-0.040440,7.135185e-02,-0.015398,-0.068789,-0.030801,-0.052920,-1.042945e-02,-0.004271,-0.030846
2023-10-31,-0.004152,0.016636,-3.132471e-03,-0.059265,-0.063625,0.031277,0.014073,1.053937e-02,-0.052335,-0.034288
2023-11-30,0.076635,-0.038423,-1.536009e-02,0.129296,0.009148,0.068968,0.083216,5.337423e-02,0.056316,0.096380


In [5]:
# Create training and test data for the prediction model.

X = []
Y = []

max_lag = 3
n_rows = len(return_matrix)

n = max_lag
while n < n_rows:
    # Get X
    X_row = (return_matrix[n - max_lag:n]).values.flatten()
    X.append(X_row)

    # Get Y
    Y_row = (return_matrix.iloc[n]).values.flatten()
    Y.append(Y_row)
    n = n + 1

X = np.array(X)
Y = np.array(Y)

print(X.shape)
print(Y.shape)

(286, 30)
(286, 10)


In [6]:
test_size = 12                          # 12 months (last year) for testing, rest for training

train_size = int(len(X) - test_size)

X_train = X[:train_size]
X_test = X[train_size:]

Y_train = Y[:train_size]
Y_test = Y[train_size:]

len(Y_train), len(Y_test)

(274, 12)

In [ ]:
x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train = x_scaler.fit_transform(X_train)
X_test = x_scaler.transform(X_test)

Y_train = y_scaler.fit_transform(Y_train)
Y_test = y_scaler.transform(Y_test)

In [8]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32)

In [ ]:
batch_size = 4

train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

## Define CVaR LP and bootsrap return scenarios

In [ ]:
# bootstrap return scenarios
# WATCH OUT: need to do this for every time step in test set with partial history (up to test date).

# !!!!!!!!! Do first with the first entry in test set

# Perform bootstrap
return_array = return_matrix.values

# Number of bootstrap scenarios
num_scenarios = 1000

# Set random seed for reproducibility
np.random.seed(42)

# Sample row indices with replacement
sample_indices = np.random.choice(return_array.shape[0], size=num_scenarios, replace=True)

# Generate bootstrapped scenario matrix
scenario_matrix = return_array[sample_indices, :]

loss_matrix = -scenario_matrix

print(scenario_matrix.shape)

(1000, 10)


In [17]:
# Define global constants for cvxpy

# scenario_matrix (S scenarios × N assets)
S, N = scenario_matrix.shape

# Empirical mean from scenarios
mu = scenario_matrix.mean(axis=0)

print(S, N)

1000 10


In [ ]:
# x_pert = solve_cvar(mu_pert, L_np)

## Implement Model

In [11]:
class VARasNN(nn.Module):

    def __init__(self,input_dim,output_dim):

        super().__init__()

        self.linear = nn.Linear(
            input_dim,
            output_dim
        )

    def forward(self,x):

        return self.linear(x)

In [12]:
input_dim = X_train.shape[1]
output_dim = Y_train.shape[1]

model = VARasNN(input_dim, output_dim)

print(model)

VARasNN(
  (linear): Linear(in_features=30, out_features=10, bias=True)
)


In [14]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [ ]:
# Custom backward pass for decision focused learning

class CustomLinear(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, weight, bias):
        ctx.save_for_backward(input, weight)
        output = input.mm(weight.t())
        output += bias.unsqueeze(0).expand_as(output)
        return output
 
    @staticmethod
    def backward(ctx, grad_output):
        input, weight = ctx.saved_tensors
        grad_input = grad_output.mm(weight)
        grad_weight = grad_output.t().mm(input)
        grad_bias = grad_output.sum(0)
        return grad_input, grad_weight, grad_bias
 
 
# Create tensors
input = torch.randn(10, 5, requires_grad=True)
weight = torch.randn(3, 5, requires_grad=True)
bias = torch.randn(3, requires_grad=True)
 
# Apply the custom linear function
linear = CustomLinear.apply
output = linear(input, weight, bias)
 
# Compute the gradients
output.sum().backward()
 
print("Gradients:")
print(f"Input: {input.grad}")
print(f"Weight: {weight.grad}")
print(f"Bias: {bias.grad}")

In [29]:
n_epochs = 100

for epoch in range(n_epochs):

    model.train()

    epoch_loss = 0

    for X_batch, Y_batch in train_loader:

        # forward pass
        predictions = model(X_batch)

        loss = criterion(predictions, Y_batch)

        # backward pass
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)

    print(f"Epoch {epoch+1}: {avg_loss:.6f}")

Epoch 1: 0.913965
Epoch 2: 0.907540
Epoch 3: 0.900972
Epoch 4: 0.890630
Epoch 5: 0.885938
Epoch 6: 0.884363
Epoch 7: 0.889877
Epoch 8: 0.874833
Epoch 9: 0.876031
Epoch 10: 0.871138
Epoch 11: 0.869393
Epoch 12: 0.872001
Epoch 13: 0.868006
Epoch 14: 0.875676
Epoch 15: 0.866168
Epoch 16: 0.860249
Epoch 17: 0.866285
Epoch 18: 0.860932
Epoch 19: 0.861116
Epoch 20: 0.860279
Epoch 21: 0.862782
Epoch 22: 0.859558
Epoch 23: 0.860817
Epoch 24: 0.859665
Epoch 25: 0.858246
Epoch 26: 0.861638
Epoch 27: 0.861741
Epoch 28: 0.858323
Epoch 29: 0.862851
Epoch 30: 0.859693
Epoch 31: 0.857156
Epoch 32: 0.857972
Epoch 33: 0.855176
Epoch 34: 0.858880
Epoch 35: 0.858415
Epoch 36: 0.863301
Epoch 37: 0.855144
Epoch 38: 0.866801
Epoch 39: 0.858173
Epoch 40: 0.856834
Epoch 41: 0.856647
Epoch 42: 0.863745
Epoch 43: 0.854820
Epoch 44: 0.860385
Epoch 45: 0.857438
Epoch 46: 0.861322
Epoch 47: 0.853677
Epoch 48: 0.860329
Epoch 49: 0.855138
Epoch 50: 0.860461
Epoch 51: 0.854721
Epoch 52: 0.866098
Epoch 53: 0.856849
Ep

In [30]:
model.eval()

with torch.no_grad():

    preds_scaled = model(X_test_tensor).numpy()

preds = y_scaler.inverse_transform(preds_scaled)

In [31]:
preds = y_scaler.inverse_transform(preds_scaled)
Y_true = y_scaler.inverse_transform(Y_test_tensor.numpy())

In [32]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(Y_true, preds)
mae = mean_absolute_error(Y_true, preds)
r2  = r2_score(Y_true, preds)

print("GLOBAL STATS")
print("MSE:", mse)
print("MAE:", mae)
print("R2 :", r2)

GLOBAL STATS
MSE: 0.006941961590200663
MAE: 0.06680236756801605
R2 : -0.7894545793533325
